<a id="shockley"></a>

$$
\text{二极管 (Diode) 与肖克利方程 (Shockley Diode Equation)} \\
I = I_s \left(e^{\frac{V}{n V_T}} - 1\right) \\
I_s \text{: 反向饱和电流 (Reverse Saturation Current, 约 } 10^{-12} \text{ A)}, \ n \text{: 理想因子 (Ideality Factor, 1~2)}, \ V_T = \frac{kT}{q} \text{: 热电压 (Thermal Voltage)} \\ \quad \\
\text{推导思路 (玻尔兹曼分布 Boltzmann Distribution): PN 结有一个势垒 } qV_{bi} \text{, 能量高于势垒的载流子比例} \propto e^{-\frac{E}{kT}} \\
\text{外加正向电压 V 让势垒降低到 } q(V_{bi} - V) \text{, 能越过势垒的扩散电流} \propto e^{-\frac{q(V_{bi} - V)}{kT}} = \underbrace{e^{-\frac{qV_{bi}}{kT}}}_{\text{常数}} e^{\frac{qV}{kT}} \\
\text{反方向的漂移电流 (少数载流子) 与 V 无关, 记为 } I_s \text{; V = 0 时两者平衡, 净电流为 0:} \\
I = I_s e^{\frac{qV}{kT}} - I_s = I_s \left(e^{\frac{V}{V_T}} - 1\right) \quad (\text{实际器件用 n 修正}) \\ \quad \\
\text{性质:} \\
\text{1. 正向 (} V \gg V_T \text{): } I \approx I_s e^{V / nV_T} \text{, 电流每增大 10 倍, 电压只增加 } nV_T \ln 10 \approx 60 \text{ mV (n = 1, 室温)} \\
\quad \text{(由 } \frac{I_2}{I_1} = e^{(V_2 - V_1)/nV_T} = 10 \implies V_2 - V_1 = nV_T \ln 10 \text{), 所以硅二极管导通电压近似恒定 0.6~0.7 V} \\
\text{2. 反向 (} V \ll -V_T \text{): } I \approx -I_s \text{, 几乎不导电 (单向导电性)} \\
\text{3. 室温 T = 300 K: } V_T = \frac{1.380649 \times 10^{-23} \times 300}{1.602177 \times 10^{-19}} \approx 25.85 \text{ mV}
$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import brentq
from scipy.special import lambertw
from scipy.integrate import solve_ivp

np.set_printoptions(linewidth=np.inf)
np.set_printoptions(suppress=True)

def plot_nicely():
    for ax in plt.gcf().get_axes():
        ax.grid(True, which="both", alpha=0.3)
        if ax.get_legend_handles_labels()[0]:
            ax.legend(fontsize='small')
    plt.tight_layout()

k_B, q_e = 1.380649e-23, 1.602176634e-19
V_T = k_B * 300 / q_e
I_s, n_ideal = 1e-12, 1.0
nVT = n_ideal * V_T
print(f'V_T (300 K) = {V_T * 1000:.4f} mV')

def diode_current(V):
    return I_s * (np.exp(V / nVT) - 1)

for I_target in [1e-6, 1e-5, 1e-4, 1e-3, 1e-2]:
    V_needed = nVT * np.log(I_target / I_s + 1)
    print(f'I = {I_target:.0e} A 需要 V = {V_needed:.4f} V')
print(f'每 10 倍电流增加 nV_T ln10 = {nVT * np.log(10) * 1000:.2f} mV')
print(f'反向 -5 V 时电流 = {diode_current(-5.0):.3e} A (≈ -I_s)')

V = np.linspace(-0.3, 0.75, 500)
fig, axs = plt.subplots(1, 2, figsize=(8, 3))
axs[0].plot(V, diode_current(V) * 1000, label='I (mA)')
axs[0].set_ylim(-1, 20)
axs[0].set_xlabel('V (V)')
axs[1].semilogy(V[V > 0.05], diode_current(V[V > 0.05]), label='I (A), log scale')
axs[1].set_xlabel('V (V)')
plot_nicely()
plt.show()

<a id="operating-point"></a>

$$
\text{工作点 (Operating Point): 电压源 } V_s \text{ 串联电阻 R 和二极管} \\
\text{KVL: } V_s = I R + V_D \text{, 二极管: } I = I_s(e^{V_D / nV_T} - 1) \text{, 消去 I 得到关于 } V_D \text{ 的非线性方程} \\
g(V_D) = I_s\left(e^{V_D / nV_T} - 1\right) - \frac{V_s - V_D}{R} = 0 \\ \quad \\
\text{1. 牛顿法: 用切线近似 g, 迭代 } V_{k+1} = V_k - \frac{g(V_k)}{g'(V_k)}, \quad g'(V) = \frac{I_s}{nV_T} e^{V / nV_T} + \frac{1}{R} \\ \quad \\
\text{2. 初始值很重要: 指数函数很陡, 从左边猜 (} g < 0 \text{) 会被切线甩到很远的右边, 之后每步只能退回约 } nV_T \text{, 甚至 } e^{V/nV_T} \text{ 溢出} \\
g''(V) = \frac{I_s}{(nV_T)^2} e^{V/nV_T} > 0 \text{, g 单调递增且是凸函数 (convex)} \\
\text{凸函数的切线在函数图像下方: 若从根的右边 } (g(V_0) > 0) \text{ 出发, 切线零点 } V_1 \text{ 满足 } g(V_1) \geq 0 \text{ 且 } V_1 < V_0 \\
\implies \text{迭代单调递减地收敛到根, 不会冲过头} \\
\text{取 } V_0 = nV_T \ln\left(\frac{V_s}{R I_s} + 1\right) \text{ (假设全部 } \frac{V_s}{R} \text{ 都流过二极管时的电压), 则 } g(V_0) = \frac{V_s}{R} - \frac{V_s - V_0}{R} = \frac{V_0}{R} > 0 \\ \quad \\
\text{3. 闭式解 (朗伯 W 函数 Lambert W, 定义为 } W(z) e^{W(z)} = z \text{ 的解):} \\
V_D = V_s - IR \text{ 代入: } I + I_s = I_s e^{(V_s - IR)/nV_T} \\
\text{令 } u = \frac{(I + I_s) R}{nV_T} \text{, 则 } \frac{IR}{nV_T} = u - \frac{I_s R}{nV_T} \text{, 两边乘 } \frac{R}{nV_T} e^{u - I_s R / nV_T} \text{ 整理:} \\
u e^{u} = \frac{I_s R}{nV_T} e^{\frac{V_s + I_s R}{nV_T}} \implies u = W\left(\frac{I_s R}{nV_T} e^{\frac{V_s + I_s R}{nV_T}}\right), \quad I = \frac{nV_T}{R} u - I_s
$$

数学基础: [牛顿法与二次收敛](../math_basics/calculus.ipynb#newton) · [极值的二阶条件 (凸性)](../math_basics/calculus.ipynb#extremum)

In [ ]:
Vs, R = 5.0, 1e3

g = lambda V: diode_current(V) - (Vs - V) / R
dg = lambda V: I_s / nVT * np.exp(V / nVT) + 1 / R

V_k = nVT * np.log(Vs / (R * I_s) + 1)
history = [V_k]
for _ in range(8):
    V_k = V_k - g(V_k) / dg(V_k)
    history.append(V_k)
V_newton = history[-1]

V_brent = brentq(g, 0.0, Vs, xtol=1e-15)
I_lambert = nVT / R * lambertw(I_s * R / nVT * np.exp((Vs + I_s * R) / nVT)).real - I_s
V_lambert = Vs - I_lambert * R

print('牛顿法迭代 (单调递减):')
for k, Vh in enumerate(history[:7]):
    print(f'  k = {k}: V = {Vh:.15f}, 误差 = {abs(Vh - V_brent):.2e}, g(V) = {g(Vh):+.3e}')
print(f'牛顿法 V_D = {V_newton:.12f} V, brentq = {V_brent:.12f} V, Lambert W = {V_lambert:.12f} V')
print(f'工作点电流 I = {(Vs - V_newton) / R * 1000:.6f} mA, 二极管方程 I = {diode_current(V_newton) * 1000:.6f} mA')

# 从左边 (g < 0) 出发的问题: 第一步被甩到很远的右边, 之后每一步只能退回约 nV_T (指数函数的切线零点),
# 要上百次迭代才能回来; 起点再靠左一点, 第一步就会超过 e^709 直接溢出
for V_bad in [0.2, 0.5]:
    V_iter, steps = V_bad, 0
    while abs(g(V_iter)) > 1e-15 and steps < 1000:
        V_iter = V_iter - g(V_iter) / dg(V_iter)
        steps += 1
        if steps == 1:
            V_first = V_iter
    print(f'从 V0 = {V_bad} 出发: 第一步跳到 V1 = {V_first:.3f} V, 共迭代 {steps} 次收敛 (从右侧初始值出发只需 4 次)')

# 负载线 (Load Line)
V_plot = np.linspace(0, 1.0, 400)
plt.figure(figsize=(4, 3))
plt.plot(V_plot, diode_current(V_plot) * 1000, label='diode I(V)')
for Vs_line in [2.0, 5.0]:
    plt.plot(V_plot, (Vs_line - V_plot) / R * 1000, '--', label=f'load line Vs = {Vs_line} V')
plt.plot(V_newton, (Vs - V_newton) / R * 1000, 'ko', label='operating point')
plt.ylim(0, 6)
plt.xlabel('V_D (V)')
plt.ylabel('I (mA)')
plot_nicely()
plt.show()

<a id="small-signal"></a>

$$
\text{小信号模型 (Small-Signal Model): 在工作点附近把二极管线性化} \\
\text{一阶泰勒展开: } I(V_Q + v) \approx I(V_Q) + \frac{dI}{dV}\bigg|_{V_Q} v = I_Q + \frac{v}{r_d} \\
\frac{dI}{dV} = \frac{I_s}{nV_T} e^{V / nV_T} = \frac{I + I_s}{nV_T} \implies r_d = \frac{nV_T}{I_Q + I_s} \approx \frac{nV_T}{I_Q} \quad (\text{动态电阻 Dynamic Resistance}) \\
\text{例: } I_Q = 4.4 \text{ mA 时 } r_d \approx \frac{25.85}{4.4} \approx 5.9 \ \Omega \text{, 电流越大, 动态电阻越小} \\ \quad \\
\text{叠加一个小的交流信号 } v_s(t) = V_s + v_{ac}\sin\omega t \text{: 直流决定工作点, 交流部分看成 } v_{ac} \text{ 加在 } R + r_d \text{ 上} \\
i_{ac} \approx \frac{v_{ac}}{R + r_d} \text{, 二极管上的交流电压 } v_d \approx v_{ac}\frac{r_d}{R + r_d} \\
\text{二阶项 } \frac{1}{2} I''(V_Q) v^2 \text{ 带来失真 (二次谐波), 信号越大越明显}
$$

数学基础: [泰勒展开 (线性化)](../math_basics/calculus.ipynb#taylor) · [链式法则](../math_basics/calculus.ipynb#derivative-rules)

In [ ]:
def solve_vd(vs):
    # 向量化的牛顿法, 正电压用上面推导的右侧初始值, 负电压二极管截止, 直接从 V_D = v_s 出发
    vs = np.atleast_1d(np.asarray(vs, dtype=float))
    V = np.where(vs > 0, nVT * np.log(np.maximum(vs, 0) / (R * I_s) + 1), vs)
    for _ in range(60):
        gV = I_s * (np.exp(V / nVT) - 1) - (vs - V) / R
        V = V - gV / (I_s / nVT * np.exp(V / nVT) + 1 / R)
    return V

I_Q = (Vs - V_newton) / R
r_d = nVT / (I_Q + I_s)
dI_numeric = (diode_current(V_newton + 1e-6) - diode_current(V_newton - 1e-6)) / 2e-6
print(f'r_d = nV_T / (I_Q + I_s) = {r_d:.6f} ohm, 数值 1 / (dI/dV) = {1 / dI_numeric:.6f} ohm')

f_ac = 1000.0
t = np.linspace(0, 2 / f_ac, 2001)
for v_ac in [0.01, 0.5, 2.0]:
    vs_t = Vs + v_ac * np.sin(2 * np.pi * f_ac * t)
    i_exact = (vs_t - solve_vd(vs_t)) / R
    i_small = I_Q + v_ac * np.sin(2 * np.pi * f_ac * t) / (R + r_d)
    print(f'v_ac = {v_ac:>4} V: 小信号模型最大相对误差 = {np.max(np.abs(i_exact - i_small)) / (v_ac / (R + r_d)) * 100:.4f}% (相对交流电流振幅)')

vd_check = solve_vd(np.array([-3.0, 0.3, 2.0, 5.0]))
print(f'向量化牛顿法 V_D = {np.round(vd_check, 6)}, brentq 对照 = {np.round([brentq(lambda V, s=s: I_s * (np.exp(V / nVT) - 1) - (s - V) / R, -5, 5) for s in [-3.0, 0.3, 2.0, 5.0]], 6)}')

<a id="rectifier"></a>

$$
\text{半波整流 (Half-Wave Rectifier): 交流电源 } v_s = V_p \sin\omega t \text{, 二极管串联负载 R} \\
\text{正半周二极管导通, } v_{out} = v_s - V_D \approx v_s - 0.6 \text{ V; 负半周截止, } v_{out} \approx 0 \text{ (每个时刻解一次上面的非线性方程)} \\ \quad \\
\text{加滤波电容 C 与负载并联: 电容电压 v 满足 (KCL: 二极管电流 = 电容电流 + 负载电流)} \\
C \frac{dv}{dt} = I_s\left(e^{\frac{v_s(t) - v}{nV_T}} - 1\right) - \frac{v}{R} \\
\text{二极管导通的瞬间电流极大 (指数), 方程是刚性的 (Stiff), 用隐式方法 (Radau) 求解} \\ \quad \\
\text{纹波电压 (Ripple Voltage) 估计: 峰值后二极管截止, 电容通过 R 放电, 大约放电一个周期 T 后再被充电} \\
v(t) = V_{peak} e^{-t/RC} \xrightarrow{\text{一阶泰勒 } e^{-x} \approx 1 - x} V_{peak}\left(1 - \frac{T}{RC}\right) \implies \Delta V \approx \frac{V_{peak} T}{RC} = \frac{V_{peak}}{f R C} = \frac{I_{load}}{f C} \\
\text{(实际放电时间比 T 稍短, 所以这是一个偏大的估计; C 越大纹波越小)}
$$

数学基础: [泰勒展开](../math_basics/calculus.ipynb#taylor) · [ODE 数值解](../math_basics/calculus.ipynb#euler-rk4) · 电路: [RC 放电](rc_rl_circuit.ipynb#rc-discharging)

In [ ]:
Vp, f_line, R_load = 10.0, 50.0, 1e3
R = R_load                                   # solve_vd 使用全局 R
w_line = 2 * np.pi * f_line
t = np.linspace(0, 0.06, 6001)
vs_t = Vp * np.sin(w_line * t)

# 1. 无电容: 每个时刻解非线性方程
v_out = vs_t - solve_vd(vs_t)
print(f'无电容: 输出最大值 = {v_out.max():.4f} V (峰值 - 二极管压降 {Vp - v_out.max():.4f} V), 最小值 = {v_out.min():.2e} V')

# 2. 有电容: 刚性 ODE
Cap = 100e-6
def rect_rhs(t, v):
    vs = Vp * np.sin(w_line * t)
    return (I_s * (np.exp(np.minimum((vs - v) / nVT, 700)) - 1) - v / R_load) / Cap

sol = solve_ivp(rect_rhs, [0, t[-1]], [0.0], t_eval=t, method='Radau', rtol=1e-8, atol=1e-10, max_step=1e-4)
v_cap = sol.y[0]
late = t > 0.02
ripple = v_cap[late].max() - v_cap[late].min()
V_peak = v_cap[late].max()
print(f'有电容 C = {Cap * 1e6:.0f} uF: 纹波 数值 = {ripple:.4f} V, 估计 V_peak / (f R C) = {V_peak / (f_line * R_load * Cap):.4f} V (偏大的估计)')

plt.figure(figsize=(4, 3))
plt.plot(t * 1000, vs_t, 'gray', alpha=0.5, label='v_s')
plt.plot(t * 1000, v_out, label='half-wave output')
plt.plot(t * 1000, v_cap, label=f'with C = {Cap * 1e6:.0f} uF')
plt.xlabel('t (ms)')
plt.ylabel('V')
plot_nicely()
plt.show()